# 10 · Rules as data: expressions

**The problem.** Marketing runs promotions with eligibility rules: "65 or older, with an email address on file",
"between 18 and 65", "a VIP, or someone spending over a limit". An admin screen edits the rules, the back end (Python)
applies them to contacts, and the web front end (TypeScript) shows who qualifies. So a rule has to be *stored* with the
rest of the data, *sent* between programs, and *evaluated* the same way by both.

A Python function can do none of that: it can't be saved, and TypeScript can't run it. An **expression** can. It's
data with a schema, like everything else in this framework. This case study follows an expression through its life:
building it, evaluating it, saving and loading it, and walking it to analyze or rewrite it.

In Python you write an expression as an ordinary lambda, and `Expressions.from_` reads it. The fluent notation, which
the TypeScript tutorial uses throughout, is in the appendix.

In [1]:
from mbse_schemas.Framework import Evaluators, Expressions, JSON, Proxies, Schemas

Contact = (
    Schemas.OfObject.Builder()
    .properties(
        lambda prop: prop.name('name').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('age').of(lambda t: t.as_native(int)),
        lambda prop: prop.name('email').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('vip').of(lambda t: t.as_native(bool)),
    )
    .create()
)
Proxies.register('Contact', Contact)
NewContact = Proxies.Builders.Contact

ada = NewContact().name('Ada').age(70).email('ada@example.com').create()
bob = NewContact().name('Bob').age(40).vip(True).create()
cy = NewContact().name('Cy').email('cy@example.com').create()   # no age on file
people = [ada, bob, cy]


def show(expression):
    """An expression as function-call text. Section 4 explains how it works."""
    e = Expressions.OfAny.resolve(expression)
    if isinstance(e, Expressions.OfLiteral.Data):
        return repr(e.value)
    if isinstance(e, Expressions.OfVariable.Data):
        return e.name
    if isinstance(e, Expressions.OfLet.Data):
        return f'let {e.name} = {show(e.value)} in {show(e.body)}'
    return f"{e.name}({', '.join(show(argument) for argument in e.arguments)})"

## 1. Building an expression

`Expressions.from_` reads a function's *source*; it never calls it. Each parameter becomes a variable, and the body
becomes operations from the core vocabulary:

In [2]:
senior = Expressions.from_(lambda contact: contact.age >= 65 and contact.email is not None)
print(show(senior))

and(ge(get(contact, 'age'), 65), has(contact, 'email'))


`contact.age` reads a property (`get`), `>=` is `ge`, `and` is `and`, and `is not None` asks whether a property
is present (`has`). The result is a `Term`: a handle on the expression's data, accepted wherever an expression is.

Chained comparisons, several parameters and names from the surrounding code work as you'd expect. A name that isn't
a parameter, like `MIN_SPEND` here, is read *once*, when `from_` runs, and becomes a literal: the expression doesn't
refer back to your program.

In [3]:
working_age = Expressions.from_(lambda contact: 18 <= contact.age < 65)
MIN_SPEND = 1000
big_spender = Expressions.from_(lambda contact, spend: contact.vip or spend > MIN_SPEND)
print(show(working_age))
print(show(big_spender))

and(le(18, get(contact, 'age')), lt(get(contact, 'age'), 65))
or(get(contact, 'vip'), gt(spend, 1000))


To name an intermediate value, call a one-parameter lambda. That's a **let**: the name is bound to the value inside
the body, and nowhere else.

In [4]:
sixties = Expressions.from_(lambda contact: (lambda age: age >= 60 and age < 70)(contact.age))
print(show(sixties))

let age = get(contact, 'age') in and(ge(age, 60), lt(age, 70))


Only what maps onto the core vocabulary is accepted, and anything else is refused when you build the rule, not
when it runs:

In [5]:
try:
    Expressions.from_(lambda contact: contact.age / 10 > 6)
except ValueError as error:
    print(error)

cannot convert 'contact.age / 10': not supported in an expression


## 2. Evaluating an expression

`Evaluators.OfAny(expression, scope)` computes the value, with the variables in `scope` bound:

In [6]:
for person in people:
    print(person.name, Evaluators.OfAny(senior, {'contact': person}))

Ada True
Bob False
Cy None


Ada qualifies and Bob doesn't. Cy's answer is `None`: her age isn't on file, so whether she is 65 or older is
**unknown**. The lambda itself would have raised `AttributeError` reading `cy.age`. An expression answers instead,
and says it doesn't know.

Unknowns combine with *three-valued logic*: `False and unknown` is `False`, `True or unknown` is `True`, and otherwise
unknown stays unknown. So a rule can still decide when enough is known:

In [7]:
reachable = Expressions.from_(lambda contact: contact.email is not None or contact.age >= 65)
[Evaluators.OfAny(reachable, {'contact': person}) for person in people]

[True, False, True]

A rule with two parameters needs both bound. Bob is a VIP, so what he spends doesn't matter; Ada's VIP flag is
absent, so her answer depends on the spend:

In [8]:
print(Evaluators.OfAny(big_spender, {'contact': bob, 'spend': 50}))
print(Evaluators.OfAny(big_spender, {'contact': ada, 'spend': 1500}))
print(Evaluators.OfAny(big_spender, {'contact': ada, 'spend': 50}))

True
True
None


**No coercion.** As everywhere in this framework (case study 7), an `int` and a `float` are different kinds of value.
Comparing them is unknown rather than a guess, which is also what a TypeScript program comparing an integer with a
number sees:

In [9]:
Evaluators.OfAny(Expressions.from_(lambda contact: contact.age >= 65.0), {'contact': ada})

Rules can be checked before they run. `validate()` reports what evaluation would raise: variables nothing binds,
wrong numbers of arguments, and, with `core=True`, operations outside the core vocabulary:

In [10]:
print(big_spender.data.validate(bound={'contact', 'spend'}, core=True))
print(big_spender.data.validate(bound={'contact'}))

[]
["argument 1: argument 0: variable 'spend' is not bound"]


## 3. Saving and loading an expression

An expression's data is `Visitable`, and every kind has a meta-schema, so it saves like any object graph (case study
6):

In [11]:
text = JSON.ToJSON.Reachable(Expressions.OfOperation.Schema, senior.data)
print(text)

{"root": "s0", "objects": {"s0": {"kind": "operation", "name": "and", "arguments": [{"argument": {"$ref": "s1", "$schema": "Expressions.OfOperation"}, "index": 0}, {"argument": {"$ref": "s2", "$schema": "Expressions.OfOperation"}, "index": 1}]}, "s1": {"kind": "operation", "name": "ge", "arguments": [{"argument": {"$ref": "s3", "$schema": "Expressions.OfOperation"}, "index": 0}, {"argument": {"$ref": "s4", "$schema": "Expressions.OfLiteral"}, "index": 1}]}, "s2": {"kind": "operation", "name": "has", "arguments": [{"argument": {"$ref": "s5", "$schema": "Expressions.OfVariable"}, "index": 0}, {"argument": {"$ref": "s6", "$schema": "Expressions.OfLiteral"}, "index": 1}]}, "s3": {"kind": "operation", "name": "get", "arguments": [{"argument": {"$ref": "s5", "$schema": "Expressions.OfVariable"}, "index": 0}, {"argument": {"$ref": "s7", "$schema": "Expressions.OfLiteral"}, "index": 1}]}, "s4": {"kind": "literal", "int": 65}, "s5": {"kind": "variable", "name": "contact"}, "s6": {"kind": "liter

Each object carries a `kind` tag, arguments are entries with an `index`, and the variable `contact`, used twice, is
written once. Nothing in the text is Python: the TypeScript front end reads the same JSON and rebuilds the same
expression. Loading it here uses `Expressions.Builders`, which build expression data rather than proxies:

In [12]:
loaded = JSON.FromJSON(Expressions.Builders).Reachable(Expressions.OfOperation.Schema, text)
print(show(loaded))
print(JSON.ToJSON.Reachable(Expressions.OfOperation.Schema, loaded) == text)
print([Evaluators.OfAny(loaded, {'contact': person}) for person in people])

and(ge(get(contact, 'age'), 65), has(contact, 'email'))
True
[True, False, None]


## 4. Traversing an expression for analysis

An expression is a small tree of four kinds of data, each with plain fields:

| Kind | Fields |
|---|---|
| `Expressions.OfLiteral.Data` | `value` |
| `Expressions.OfVariable.Data` | `name` |
| `Expressions.OfOperation.Data` | `name`, `arguments` |
| `Expressions.OfLet.Data` | `name`, `value`, `body` |

`show`, above, is such a traversal. Here is a more useful one: which properties does a rule read? The admin screen
can use it to load only those fields, or to warn when a rule reads a property the schema doesn't have.

In [13]:
def reads(expression):
    """The (variable, property) pairs an expression reads with get or has."""
    e = Expressions.OfAny.resolve(expression)
    if isinstance(e, Expressions.OfLet.Data):
        return reads(e.value) | reads(e.body)
    if not isinstance(e, Expressions.OfOperation.Data):
        return set()
    found = set().union(*(reads(argument) for argument in e.arguments))
    if e.name in ('get', 'has') and isinstance(e.arguments[0], Expressions.OfVariable.Data):
        found.add((e.arguments[0].name, e.arguments[1].value))
    return found


for rule in [senior, big_spender, sixties]:
    print(show(rule), '->', sorted(reads(rule)))

and(ge(get(contact, 'age'), 65), has(contact, 'email')) -> [('contact', 'age'), ('contact', 'email')]
or(get(contact, 'vip'), gt(spend, 1000)) -> [('contact', 'vip')]
let age = get(contact, 'age') in and(ge(age, 60), lt(age, 70)) -> [('contact', 'age')]


Scoping matters in a traversal. A variable bound by a let isn't a parameter of the rule; `free_variables` keeps
track of the names each let binds:

In [14]:
def free_variables(expression, bound=frozenset()):
    """The variables an expression needs bound by whoever evaluates it."""
    e = Expressions.OfAny.resolve(expression)
    if isinstance(e, Expressions.OfVariable.Data):
        return {e.name} - bound
    if isinstance(e, Expressions.OfLet.Data):
        return free_variables(e.value, bound) | free_variables(e.body, bound | {e.name})
    if isinstance(e, Expressions.OfOperation.Data):
        return set().union(*(free_variables(argument, bound) for argument in e.arguments))
    return set()


print(sorted(free_variables(big_spender)))
print(sorted(free_variables(sixties)))

['contact', 'spend']
['contact']


`validate(bound=...)` makes the same check. Writing it yourself shows the pattern every analysis follows: one case
per kind, recursing into arguments, lets' values and lets' bodies.

## 5. Traversing an expression for substitution

A rule can be written once as a *template* and specialized later. Here the age limit is a parameter; a campaign fixes
it by replacing the variable with a literal. Substitution rebuilds the tree with builders, so the template itself is
untouched:

In [15]:
def substitute(expression, values, bound=frozenset()):
    """A copy of `expression` with the free variables named in `values` replaced by literals."""
    e = Expressions.OfAny.resolve(expression)
    if isinstance(e, Expressions.OfVariable.Data):
        return Expressions.OfLiteral.resolve(values[e.name]) if e.name in values and e.name not in bound else e
    if isinstance(e, Expressions.OfLet.Data):
        return (Expressions.OfLet.Builder().name(e.name).value(substitute(e.value, values, bound))
                .body(substitute(e.body, values, bound | {e.name})).create())
    if isinstance(e, Expressions.OfOperation.Data):
        return Expressions.OfOperation.Builder().name(e.name).arguments(
            *(substitute(argument, values, bound) for argument in e.arguments)).create()
    return e


at_least = Expressions.from_(lambda contact, limit: contact.age >= limit)
senior_campaign = substitute(at_least, {'limit': 65})
print(show(at_least), '->', show(senior_campaign))
print(sorted(free_variables(senior_campaign)), Evaluators.OfAny(senior_campaign, {'contact': ada}))

ge(get(contact, 'age'), limit) -> ge(get(contact, 'age'), 65)
['contact'] True


The `bound` set stops substitution at a let that binds the same name, just as evaluation would:

In [16]:
shadowed = Expressions.from_(lambda contact, limit: (lambda limit: contact.age >= limit)(18))
print(show(substitute(shadowed, {'limit': 65})))

let limit = 18 in ge(get(contact, 'age'), limit)


Parts that don't change, such as literals and variables that aren't replaced, are shared between the template
and the copy rather than copied. Expressions are never changed in place, so sharing is safe.

## What's next

Expressions are what union discriminators are made of (case study 9): each branch's predicate is evaluated with
`this` bound to the value, and the first that is `True` chooses the branch. Choosing branches that way, and the
collection operations (`count`, `in`, `all`, `any`) that constraints like "at least one phone" need, are designed but
not built yet.

## Appendix: building expressions with methods and builders

`from_` is a Python convenience. Underneath, and in TypeScript, expressions are written with **terms**: methods that
build the same data. A variable is a term, `.name` reads a property, and methods build the operations (`eq`, `ne`,
`lt`, `le`, `gt`, `ge`, `and_`, `or_`, `not_`, `implies`, `add`, `sub`, `mul`, `neg`, `has`, `get`):

In [17]:
contact = Expressions.variable('contact')
by_hand = contact.age.ge(65).and_(contact.has('email'))
print(show(by_hand))
print(JSON.ToJSON.Reachable(Expressions.OfOperation.Schema, by_hand.data) == text)

and(ge(get(contact, 'age'), 65), has(contact, 'email'))
True


The JSON is identical to the one `from_` produced in section 3: the two notations build the same data.
`Expressions.literal`, `Expressions.let_` and `Expressions.operation` (for operations outside the core vocabulary)
start other terms.

Each kind also has a builder, like every other element of the framework, finalized by `create()`, `clone()` or
`update()`. Builders are the lowest level; section 5 used them because they take data apart and put it together:

In [18]:
age = Expressions.OfOperation.Builder().name('get').arguments(contact, 'age').create()
built = Expressions.OfOperation.Builder().name('and').arguments(
    Expressions.OfOperation.Builder().name('ge').arguments(age, 65).create(),
    Expressions.OfOperation.Builder().name('has').arguments(contact, 'email').create(),
).create()
print(show(built))
print(JSON.ToJSON.Reachable(Expressions.OfOperation.Schema, built) == text)

and(ge(get(contact, 'age'), 65), has(contact, 'email'))
True


Use `from_` for rules you write by hand, terms when a program assembles rules, and builders when you take
expressions apart or rebuild them.

## The whole tour

| Case study | Problem | Idea |
|---|---|---|
| 1 | a contact card | schemas are values; builders; `create`/`update`/`clone`; absent is not `None` |
| 2 | an address book | relations instead of lists; untyped links; entries are shared facts |
| 3 | moving house | editing entries through builders; what `update()` replaces |
| 4 | a family tree | self-relations, cycles, `Reachable.of`; unambiguous inference |
| 5 | a price list | `unique(...)` cardinality; validation on demand |
| 6 | saving and loading | snapshots, symbols, references with schema names |
| 7 | JSON and YAML | strict text; the Norway problem; no coercion |
| 8 | tools for every schema | walking schemas; writing a visitor |
| 9 | evolving schemas | optional fields; versions; directories; variants |
| 10 | rules as data | expressions: building, evaluating, saving, analyzing and rewriting them |

The through-line: **describe data once, as data, and let every tool read that description.** Most of the ways this
framework differs from the usual approach (no lists, no mandatory fields, no automatic validation, no type
coercion, no guessing) are there so that description means the same thing to every program that reads it. Rules
are no exception: an expression is data too.